# Earnings Call Analysis

# 1. Setup

In [1]:
%pip install -q datasets transformers sentence-transformers chromadb rank_bm25 pydantic langchain-core nltk rouge-score bitsandbytes accelerate plotly streamlit pyarrow

import gc
import os
import re
import json
import math
import time
import logging
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Literal

import numpy as np
import pandas as pd
import torch
import nltk
import chromadb
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pydantic import BaseModel, Field

from datasets import load_dataset
from nltk.tokenize import sent_tokenize
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer, CrossEncoder, util
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from rouge_score import rouge_scorer

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("EarningsDemoV7")

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

DEVICE = 0 if torch.cuda.is_available() else -1
DEVICE_STR = "cuda" if DEVICE == 0 else "cpu"
print(f"Setup complete. Running on: {'GPU (CUDA)' if DEVICE == 0 else 'CPU'}")
if DEVICE == 0:
    print(f"GPU Device: {torch.cuda.get_device_name(0)} | Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
Setup complete. Running on: GPU (CUDA)
GPU Device: Tesla T4 | Total VRAM: 15.6 GB


In [2]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/TA2_Earnings_Project_v7")
else:
    BASE_DIR = Path("./TA2_Earnings_Project_v7")

DATA_DIR = BASE_DIR / "data_processed"
CHROMA_DIR = BASE_DIR / "chroma_db"
EVAL_DIR = BASE_DIR / "eval_data"
OUT_DIR = BASE_DIR / "final_outputs"

for d in [DATA_DIR, CHROMA_DIR, EVAL_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL_ID = "BAAI/bge-m3"
RERANKER_MODEL_ID = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

PREP_CHUNK_SIZE = 350
PREP_CHUNK_OVERLAP = 60
MAX_QA_CHUNK_WORDS = 600
NUM_DEMO_CALLS = 5
DEMO_SEED = 101

print(f"Project root: {BASE_DIR.resolve()}")
print(f"Demo calls: {NUM_DEMO_CALLS} | Embedder: {EMBEDDING_MODEL_ID} | LLM: {LLM_MODEL_ID}")

Project root: /content/TA2_Earnings_Project_v7
Demo calls: 5 | Embedder: BAAI/bge-m3 | LLM: Qwen/Qwen2.5-7B-Instruct


# 2. Data Loading, Safe Harbor Cleaning & 2-Stage Hybrid Parser

* **Cell 2.1:** Načtení datasetu `glopardo/sp500-earnings-transcripts`.
* **Cell 2.2:** Odstranění právních *Safe Harbor* disclaimerů z *Prepared Remarks*.
* **Cell 2.3:** Dvoufázový parser `parse_transcript` — Regex strukturální rozdělení + **DeBERTa NLI** klasifikace rolí (`Executive` vs. `Analyst`) u přepisů bez úvodního rosteru.
* **Cell 2.4:** Extrakce Q&A dvojic (`extract_qa_pairs`) včetně `has_question` pojistky proti mezimanažerským poznámkám.
* **Cell 2.5:** Turn-Aware / Q&A-Dyad chunkování (plus 500w baseline pro RAG ablaci).

##Load S&P 500 Dataset

In [3]:
ds = load_dataset("glopardo/sp500-earnings-transcripts")["train"]
print(f"Loaded S&P 500 dataset: {len(ds):,} earnings call transcripts.")
print("Columns:", ds.column_names)

README.md:   0%|          | 0.00/29.0k [00:00<?, ?B/s]

data/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  190MB            

data/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  185MB            

data/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

data/train-00002-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  187MB            

data/train-00002-of-00003.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/20681 [00:00<?, ? examples/s]

Loaded S&P 500 dataset: 20,681 earnings call transcripts.
Columns: ['ticker', 'company', 'cik', 'sector', 'industry', 'headquarters', 'founded', 'date_added', 'year', 'quarter', 'datacqtr', 'datafqtr', 'earnings_date', 'eps12mtrailing_qavg', 'eps12mtrailing_eoq', 'eps12mfwd_qavg', 'eps12mfwd_eoq', 'eps_lt', 'peforw_qavg', 'peforw_eoq', 'transcript']


##Safe Harbor Cleaning

In [4]:
SAFE_HARBOR_PATTERNS = re.compile(
    r"(?:"
    r"forward[-\s]looking statements?|"
    r"private securities litigation reform act|"
    r"safe harbor|"
    r"actual results (?:may|could|might) differ materially|"
    r"risks and uncertainties|"
    r"filings with the (?:sec|securities and exchange commission)|"
    r"form 10-k|form 10-q|"
    r"undertake no obligation to update|"
    r"non-gaap financial measures?"
    r")",
    re.IGNORECASE,
)


def remove_safe_harbor(text: str) -> str:
    if not text or not isinstance(text, str):
        return ""
    sentences = sent_tokenize(text)
    return " ".join(s for s in sentences if not SAFE_HARBOR_PATTERNS.search(s)).strip()


def make_call_id(meta: dict) -> str:
    ticker = re.sub(r"[^A-Za-z0-9_-]", "", str(meta.get("ticker", "UNK")))
    quarter = re.sub(r"[^A-Za-z0-9_-]", "", str(meta.get("datacqtr", "UNK")))
    date_part = re.sub(r"[^0-9]", "", str(meta.get("earnings_date", ""))[:10])
    return f"{ticker}_{quarter}_{date_part}" if date_part else f"{ticker}_{quarter}"

2.3 — Speaker Sections Parser

In [5]:
ROSTER_HEADER_RE = re.compile(
    r"(?:Executives|Analysts|Corporate Participants|Conference Call Participants|Call Participants)\s*:",
    re.IGNORECASE,
)

ANALYST_HEADER_RE = re.compile(
    r"(?:Analysts|Conference Call Participants|Call Participants)\s*:",
    re.IGNORECASE,
)

QA_CUE_RE = re.compile(
    r"(?:"
    r"first question|next question|"
    r"open (?:the (?:call|line|lines|floor|conference) )?(?:for|to|up for) questions|"
    r"begin (?:the )?(?:question[-\s]and[-\s]answer|q&a)|"
    r"proceed (?:with|to) (?:the )?(?:question[-\s]and[-\s]answer|q&a)|"
    r"turn (?:the call )?(?:over )?to (?:the )?(?:question[-\s]and[-\s]answer|q&a)|"
    r"ready (?:for|to take) (?:your )?questions|"
    r"now take (?:your )?(?:first )?questions?|"
    r"question[-\s]and[-\s]answer session (?:will now|begins|is now)|"
    r"our first caller|go to (?:the |our )?(?:first|next) (?:question|caller|line)"
    r")",
    re.IGNORECASE,
)

NON_SPEAKER_PREFIXES = {
    "note", "summary", "operator instructions", "question", "answer", "update",
    "fiscal", "quarter", "slide", "chart", "table", "thank you", "good morning",
    "good afternoon", "good evening", "ladies and gentlemen",
}

ROLE_LABELS = [
    "company executive discussing their own company's financials, operations, or strategy",
    "wall street analyst asking questions about the company",
]


def _clean_candidate_speakers(body_part: str) -> List[str]:
    """Detects speaker labels (e.g. 'Linsley Carruth :', 'Operator:', 'David Niederman :')."""
    raw_names = re.findall(
        r"(?:^|[.?!\n]\s*|\]\s+)([A-Z][a-zA-ZÀ-ÿ\'\.-]+(?:\s+[A-Z][a-zA-ZÀ-ÿ\'\.-]+){1,3})\s*(?:-\s*[^:\n]{1,60}?)?\s*:",
        body_part,
    )
    cleaned, seen = [], set()
    for n in raw_names:
        n_clean = re.sub(r"\s+", " ", n).strip(" .,-")
        low = n_clean.lower()
        if not n_clean or low in seen or low in NON_SPEAKER_PREFIXES or len(n_clean) < 4 or len(n_clean) > 45:
            continue
        seen.add(low)
        cleaned.append(n_clean)
    return cleaned


def _refine_inferred_roles_with_nli(df: pd.DataFrame) -> pd.DataFrame:
    """Uses already-loaded DeBERTa NLI to classify Executive vs. Analyst per speaker when no roster exists."""
    non_op = df[df["role"] != "Operator"]
    speakers = non_op["speaker"].unique().tolist()
    if not speakers:
        return df

    snippets = [
        " ".join(" ".join(non_op[non_op["speaker"] == sp]["text"].tolist()).split()[:150])
        for sp in speakers
    ]
    preds = nli_pipe(snippets, ROLE_LABELS, multi_label=False, batch_size=16)
    if isinstance(preds, dict):
        preds = [preds]

    role_map = {
        sp: ("Executive" if res["labels"][0] == ROLE_LABELS[0] else "Analyst")
        for sp, res in zip(speakers, preds)
    }
    df = df.copy()
    df.loc[df["role"] != "Operator", "role"] = df.loc[df["role"] != "Operator", "speaker"].map(role_map)

    analyst_idxs = df.index[df["role"] == "Analyst"].tolist()
    if analyst_idxs:
        df.loc[analyst_idxs[0]:, "section"] = "Q&A"
    return df


def parse_transcript(text: str, strip_safe_harbor: bool = True) -> pd.DataFrame:
    """Parses a raw earnings call transcript into chronological speaker turns (handles missing roster & missing Operator)."""
    if not text or not isinstance(text, str):
        return pd.DataFrame()

    text = text.lstrip("\ufeff").strip()
    if len(text) < 200:
        return pd.DataFrame()

    op_match = re.search(r"\bOperator\s*:", text)
    roster_part, body_part = "", text
    if op_match:
        before_op = text[: op_match.start()].strip()
        if before_op and (ROSTER_HEADER_RE.search(before_op) or (len(before_op) < 1800 and len(sent_tokenize(before_op)) <= 4)):
            roster_part = before_op
            body_part = text[op_match.start() :]

    if roster_part and ANALYST_HEADER_RE.search(roster_part):
        exec_part, analyst_part = ANALYST_HEADER_RE.split(roster_part, maxsplit=1)
    else:
        exec_part, analyst_part = roster_part, ""

    candidate_names = _clean_candidate_speakers(body_part)
    exec_lower, analyst_lower = exec_part.lower(), analyst_part.lower()

    exec_names, analyst_names = [], []
    if roster_part:
        for name in candidate_names:
            if name.lower() in exec_lower:
                exec_names.append(name)
            elif name.lower() in analyst_lower:
                analyst_names.append(name)

    inferred_mode = not (exec_names or analyst_names)
    all_detected = candidate_names if inferred_mode else (exec_names + analyst_names)
    if not all_detected:
        return pd.DataFrame()

    all_speakers = sorted(
        list(dict.fromkeys(all_detected + ["Operator", "Moderator", "Unknown Analyst", "Unknown Executive"])),
        key=len,
        reverse=True,
    )
    speaker_pattern = "|".join(re.escape(name) for name in all_speakers)
    turn_regex = re.compile(rf"(?:^|(?<=[.?!\s\]]))({speaker_pattern})(?:\s*-\s*[^:\n]{{1,60}}?)?\s*:(?:\s*:)?")

    matches = list(turn_regex.finditer(body_part))
    if len(matches) < 3:
        return pd.DataFrame()

    turns, in_qa, n_matches = [], False, len(matches)
    for i, match in enumerate(matches):
        speaker = match.group(1).strip()
        start_idx = match.end()
        end_idx = matches[i + 1].start() if i + 1 < n_matches else len(body_part)
        raw_speech = body_part[start_idx:end_idx].strip()
        if not raw_speech:
            continue

        if speaker in ("Operator", "Moderator"):
            role = "Operator"
            if i > 0 and QA_CUE_RE.search(raw_speech):
                in_qa = True
        elif inferred_mode:
            if not in_qa:
                if i >= 2 and (speaker not in exec_names) and ("?" in raw_speech) and len(raw_speech.split()) < 350:
                    in_qa = True
                    role = "Analyst"
                else:
                    if speaker not in exec_names:
                        exec_names.append(speaker)
                    role = "Executive"
                    if i >= 1 and QA_CUE_RE.search(raw_speech):
                        clean_speech = remove_safe_harbor(raw_speech) if strip_safe_harbor else raw_speech
                        if clean_speech:
                            turns.append({
                                "turn_id": len(turns) + 1, "speaker": speaker, "role": role,
                                "section": "Prepared Remarks", "word_count": len(clean_speech.split()),
                                "text": clean_speech, "raw_text": raw_speech,
                            })
                        in_qa = True
                        continue
            else:
                role = "Executive" if speaker in exec_names else "Analyst"
        else:
            if speaker in exec_names or speaker == "Unknown Executive":
                role = "Executive"
            elif speaker in analyst_names or speaker == "Unknown Analyst":
                role = "Analyst"
            else:
                role = "Operator"

        if role == "Analyst":
            in_qa = True

        section = "Q&A" if in_qa else "Prepared Remarks"
        clean_speech = remove_safe_harbor(raw_speech) if (strip_safe_harbor and section == "Prepared Remarks") else raw_speech
        if not clean_speech:
            continue

        turns.append({
            "turn_id": len(turns) + 1, "speaker": speaker, "role": role,
            "section": section, "word_count": len(clean_speech.split()),
            "text": clean_speech, "raw_text": raw_speech,
        })

    df = pd.DataFrame(turns)
    if inferred_mode and not df.empty and "nli_pipe" in globals():
        df = _refine_inferred_roles_with_nli(df)
    return df


# test
_ds_shuf42 = ds.shuffle(seed=42)
for _test_idx in [9, 14, 24]:
    _row = _ds_shuf42[_test_idx]
    _tdf = parse_transcript(_row["transcript"])
    _secs = _tdf["section"].value_counts().to_dict() if not _tdf.empty else {}
    print(f"Parser check IDX {_test_idx} ({_row['ticker']}): {len(_tdf)} turns | Sections: {_secs}")

Parser check IDX 9 (TROW): 47 turns | Sections: {'Q&A': 42, 'Prepared Remarks': 5}
Parser check IDX 14 (FIS): 75 turns | Sections: {'Q&A': 71, 'Prepared Remarks': 4}
Parser check IDX 24 (PANW): 55 turns | Sections: {'Q&A': 52, 'Prepared Remarks': 3}


## Q&A Thread & Pair Extractor

In [6]:
def extract_qa_pairs(
    turns_df: pd.DataFrame,
    min_q_words: int = 5,
    min_followup_words: int = 2,
    min_a_words: int = 5,
) -> pd.DataFrame:
    qa_turns = turns_df[turns_df["section"] == "Q&A"]
    if qa_turns.empty:
        return pd.DataFrame()

    merged = []
    for row in qa_turns.itertuples(index=False):
        txt = str(row.text).strip()
        if not txt:
            continue
        if merged and merged[-1]["role"] == row.role and merged[-1]["speaker"] == row.speaker:
            merged[-1]["text"] += " " + txt
            merged[-1]["end_turn_id"] = row.turn_id
        else:
            merged.append({
                "role": row.role, "speaker": row.speaker, "text": txt,
                "start_turn_id": row.turn_id, "end_turn_id": row.turn_id,
            })

    pairs, pair_id, thread_id = [], 1, 0
    current_thread_analyst, seen_q_keys, i = None, set(), 0

    while i < len(merged):
        turn = merged[i]
        if turn["role"] == "Operator":
            current_thread_analyst = None
            i += 1
            continue

        if turn["role"] == "Analyst":
            analyst, q_text, start_tid = turn["speaker"], turn["text"], turn["start_turn_id"]
            exec_speakers, exec_texts, end_tid = [], [], turn["end_turn_id"]
            j = i + 1
            while j < len(merged) and merged[j]["role"] == "Executive":
                exec_speakers.append(merged[j]["speaker"])
                exec_texts.append(merged[j]["text"])
                end_tid = merged[j]["end_turn_id"]
                j += 1

            a_text = " ".join(exec_texts).strip()
            is_followup = (analyst == current_thread_analyst)
            q_threshold = min_followup_words if is_followup else min_q_words
            q_key = (analyst.lower(), q_text[:100].lower())

            has_question = ("?" in q_text) or bool(
                re.search(r"\b(?:can you|could you|would you|what|how|why|wonder|curious|question|prospects)\b", q_text, re.I)
            )

            if (
                has_question
                and len(q_text.split()) >= q_threshold
                and len(a_text.split()) >= min_a_words
                and q_key not in seen_q_keys
            ):
                seen_q_keys.add(q_key)
                if not is_followup:
                    thread_id += 1
                    current_thread_analyst = analyst

                handoffs = sum(1 for k in range(1, len(exec_speakers)) if exec_speakers[k] != exec_speakers[k - 1])
                pairs.append({
                    "pair_id": pair_id, "thread_id": thread_id, "is_followup": is_followup,
                    "analyst": analyst, "question": q_text,
                    "executives": " -> ".join(dict.fromkeys(exec_speakers)),
                    "handoff_count": handoffs, "answer": a_text,
                    "start_turn_id": start_tid, "end_turn_id": end_tid,
                })
                pair_id += 1
            i = j
        else:
            i += 1

    return pd.DataFrame(pairs)

##Q&A Turns

In [7]:
def create_turn_aware_chunks(
    turns_df: pd.DataFrame,
    qa_df: pd.DataFrame,
    call_id: str,
    meta: dict,
    prep_chunk_size: int = PREP_CHUNK_SIZE,
    prep_overlap: int = PREP_CHUNK_OVERLAP,
    max_qa_words: int = MAX_QA_CHUNK_WORDS,
) -> pd.DataFrame:
    """Splits Prepared Remarks into overlapping blocks and keeps complete Q&A pairs intact."""
    chunks = []

    prep_turns = turns_df[(turns_df["section"] == "Prepared Remarks") & (turns_df["role"] == "Executive")]
    prep_text = " ".join(f"[{r.speaker}]: {r.text}" for r in prep_turns.itertuples(index=False))
    prep_words = prep_text.split()
    step = max(1, prep_chunk_size - prep_overlap)
    sec_idx = 0
    for start in range(0, len(prep_words), step):
        block = " ".join(prep_words[start : start + prep_chunk_size])
        if len(block.split()) < 25:
            continue
        chunks.append({
            "chunk_id": f"{call_id}_prep_{sec_idx}",
            "call_id": call_id,
            "ticker": str(meta.get("ticker", "UNK")),
            "company": str(meta.get("company", "UNK")),
            "quarter": str(meta.get("datacqtr", "UNK")),
            "section": "Prepared Remarks",
            "section_chunk_idx": sec_idx,
            "word_count": len(block.split()),
            "text": block,
        })
        sec_idx += 1
        if start + prep_chunk_size >= len(prep_words):
            break

    qa_idx = 0
    if not qa_df.empty:
        for r in qa_df.itertuples(index=False):
            q_header = f"[Analyst - {r.analyst}]: {r.question}"
            a_words = str(r.answer).split()
            if len(q_header.split()) + len(a_words) <= max_qa_words:
                dyad_text = f"{q_header}\n[Management - {r.executives}]: {r.answer}"
                chunks.append({
                    "chunk_id": f"{call_id}_qa_{qa_idx}",
                    "call_id": call_id,
                    "ticker": str(meta.get("ticker", "UNK")),
                    "company": str(meta.get("company", "UNK")),
                    "quarter": str(meta.get("datacqtr", "UNK")),
                    "section": "Q&A",
                    "section_chunk_idx": qa_idx,
                    "word_count": len(dyad_text.split()),
                    "text": dyad_text,
                })
                qa_idx += 1
            else:
                q_short = " ".join(str(r.question).split()[-100:])
                sub_size = max(150, max_qa_words - 120)
                for s in range(0, len(a_words), sub_size):
                    sub_a = " ".join(a_words[s : s + sub_size])
                    dyad_text = f"[Analyst - {r.analyst}]: {q_short}\n[Management - {r.executives}]: {sub_a}"
                    chunks.append({
                        "chunk_id": f"{call_id}_qa_{qa_idx}",
                        "call_id": call_id,
                        "ticker": str(meta.get("ticker", "UNK")),
                        "company": str(meta.get("company", "UNK")),
                        "quarter": str(meta.get("datacqtr", "UNK")),
                        "section": "Q&A",
                        "section_chunk_idx": qa_idx,
                        "word_count": len(dyad_text.split()),
                        "text": dyad_text,
                    })
                    qa_idx += 1

    return pd.DataFrame(chunks)


def create_fixed_chunks(
    turns_df: pd.DataFrame,
    call_id: str,
    meta: dict,
    chunk_size: int = 500,
    overlap: int = 100,
) -> pd.DataFrame:
    """Fixed 500-word baseline chunker kept strictly for the Chapter 7.2 RAG Ablation Study."""
    full_text = " ".join(f"[{r.speaker}]: {r.text}" for r in turns_df.itertuples(index=False))
    words = full_text.split()
    step = max(1, chunk_size - overlap)
    rows = []
    for idx, start in enumerate(range(0, len(words), step)):
        block = " ".join(words[start : start + chunk_size])
        if len(block.split()) < 25:
            continue
        rows.append({
            "chunk_id": f"{call_id}_fixed_{idx}",
            "call_id": call_id,
            "ticker": str(meta.get("ticker", "UNK")),
            "section": "Full Transcript",
            "section_chunk_idx": idx,
            "word_count": len(block.split()),
            "text": block,
        })
        if start + chunk_size >= len(words):
            break
    return pd.DataFrame(rows)

# 3. Phase 1 GPU — Encoder NLP Models Initialization

In [8]:
nli_pipe = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli",
    device=DEVICE,
)

finbert_pipe = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    top_k=None,
    device=DEVICE,
    truncation=True,
    max_length=512,
)

embedder = SentenceTransformer(EMBEDDING_MODEL_ID, device=DEVICE_STR)
print(f"Loaded DeBERTa-v3 NLI, ProsusAI/finbert, and {EMBEDDING_MODEL_ID} on {DEVICE_STR}.")

config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  369MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

... (25 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Loaded DeBERTa-v3 NLI, ProsusAI/finbert, and BAAI/bge-m3 on cuda.


# 4. Financial Sentiment (FinBERT), Q&A Evasion & Demo Batch Export

##Q&A Trajectory

In [9]:
def classify_finbert_net(net_sentiment: float, pos_thresh: float = 10.0, neg_thresh: float = -5.0) -> str:
    if net_sentiment >= pos_thresh:
        return "positive"
    if net_sentiment <= neg_thresh:
        return "negative"
    return "neutral"


def score_finbert_text(text: str) -> dict:
    sentences = [s for s in sent_tokenize(text) if len(s.split()) >= 4]
    if not sentences:
        return {"net_sentiment": 0.0, "label": "neutral", "n_sentences": 0}

    preds = finbert_pipe(sentences, batch_size=32)
    pos, neg = [], []
    for sent_pred in preds:
        scores = {item["label"].lower(): item["score"] for item in sent_pred}
        pos.append(scores.get("positive", 0.0))
        neg.append(scores.get("negative", 0.0))

    net_sent = round((float(np.mean(pos)) - float(np.mean(neg))) * 100, 2)
    return {
        "net_sentiment": net_sent,
        "label": classify_finbert_net(net_sent),
        "n_sentences": len(sentences),
    }


def score_finbert_qa_trajectory(chunks_df: pd.DataFrame) -> pd.DataFrame:
    qa_chunks = chunks_df[chunks_df["section"] == "Q&A"].sort_values("section_chunk_idx")
    rows = []
    for r in qa_chunks.itertuples(index=False):
        fb = score_finbert_text(r.text)
        rows.append({
            "call_id": r.call_id,
            "section_chunk_idx": int(r.section_chunk_idx),
            "finbert_net_score": fb["net_sentiment"],
            "finbert_label": fb["label"],
        })
    return pd.DataFrame(rows)

##Q&A Evasion & Semant

In [10]:
REFUSAL_REGEX = re.compile(
    r"don['’]t report|not answering it directly|wait until|won['’]t comment|"
    r"not going to give|too early to say|not prepared to|can['’]t quantify",
    re.IGNORECASE,
)

CANDIDATE_LABELS = [
    "directly answers the question with specific facts or numbers",
    "hedges, expresses uncertainty, or only partially answers the question",
    "refuses to answer, defers to a future date, or deflects the question",
]


def score_nli_chunked(q: str, a: str, chunk_words: int = 180, max_q_words: int = 120) -> float:
    q_words = q.split()
    q_trimmed = " ".join(q_words[-max_q_words:]) if len(q_words) > max_q_words else q
    a_words = a.split()
    chunks = [" ".join(a_words[i : i + chunk_words]) for i in range(0, len(a_words), chunk_words)]
    if not chunks:
        return 0.5

    contexts = [f"Analyst Question: {q_trimmed}\nExecutive Answer: {c}" for c in chunks]
    results = nli_pipe(contexts, CANDIDATE_LABELS, multi_label=False, batch_size=16)
    if isinstance(results, dict):
        results = [results]

    probs = []
    for res in results:
        scores = dict(zip(res["labels"], res["scores"]))
        probs.append((scores[CANDIDATE_LABELS[2]] * 1.0) + (scores[CANDIDATE_LABELS[1]] * 0.5))
    return float(min(probs))


def score_qa_exchange(row: pd.Series) -> pd.Series:
    q, a = row["question"], row["answer"]
    a_sentences = [s for s in sent_tokenize(a) if len(s.split()) >= 4] or [a]
    emb_q = embedder.encode(q, convert_to_tensor=True, normalize_embeddings=True)
    emb_a_sents = embedder.encode(a_sentences, convert_to_tensor=True, normalize_embeddings=True)
    semantic_alignment = float(util.cos_sim(emb_q, emb_a_sents).max())

    q_nums = len(re.findall(r"\b\d+(?:\.\d+)?%?|\$\d+", q))
    a_nums = len(re.findall(r"\b\d+(?:\.\d+)?%?|\$\d+", a))
    quant_evasion = 1.0 if (q_nums >= 2 and a_nums == 0) else 0.0

    nli_evasive_prob = score_nli_chunked(q, a)
    explicit_refusal = bool(REFUSAL_REGEX.search(a))

    evasion_score = min(
        100.0,
        round(
            (nli_evasive_prob * 55)
            + ((1.0 - max(0.0, semantic_alignment)) * 25)
            + (quant_evasion * 10)
            + (25.0 if explicit_refusal else 0.0),
            1,
        ),
    )
    if explicit_refusal or evasion_score >= 62:
        tier = "Fully Evasive"
    elif evasion_score >= 38:
        tier = "Intermediate (Hedging/Pivot)"
    else:
        tier = "Direct"

    a_fb = score_finbert_text(a)
    return pd.Series({
        "evasion_tier": tier,
        "evasion_score": evasion_score,
        "semantic_alignment": round(semantic_alignment, 3),
        "answer_finbert_net": a_fb["net_sentiment"],
        "answer_finbert_label": a_fb["label"],
        "answer_finbert_n_sents": a_fb["n_sentences"],
    })

##FinBERT Call Summary

In [11]:
HEDGING_LEXICON_RE = re.compile(
    r"\b(?:may|might|could|uncertain|uncertainty|headwind|headwinds|cautious|cautiously|"
    r"visibility|pressure|pressures|challenging|challenge|softness|volatile|volatility)\b",
    re.IGNORECASE,
)


def compute_finbert_summary_fast(turns_df: pd.DataFrame, qa_analysis_df: pd.DataFrame) -> dict:
    prep_exec_text = " ".join(
        turns_df[(turns_df["role"] == "Executive") & (turns_df["section"] == "Prepared Remarks")]["text"]
    )
    prep_fb = score_finbert_text(prep_exec_text)
    prep_n = prep_fb["n_sentences"]

    qa_weights = qa_analysis_df["answer_finbert_n_sents"].fillna(0).values if not qa_analysis_df.empty else np.array([])
    qa_n = int(qa_weights.sum())
    qa_net = (
        float(np.average(qa_analysis_df["answer_finbert_net"].values, weights=qa_weights))
        if qa_n > 0 else 0.0
    )
    total_n = prep_n + qa_n
    overall_net = ((prep_fb["net_sentiment"] * prep_n) + (qa_net * qa_n)) / total_n if total_n > 0 else 0.0

    return {
        "FINBERT_SCORE": round(overall_net, 2),
        "FINBERT_PREP_SCORE": round(prep_fb["net_sentiment"], 2),
        "FINBERT_QA_SCORE": round(qa_net, 2),
        "FINBERT_TONE_DELTA": round(qa_net - prep_fb["net_sentiment"], 2),
    }


def compute_call_card_metrics(fb_summary: dict, qa_analysis_df: pd.DataFrame) -> dict:
    """Computes transparent call-level KPIs for the Executive Dashboard header."""
    total_q = max(1, len(qa_analysis_df))
    full_ev = int((qa_analysis_df["evasion_tier"] == "Fully Evasive").sum())
    inter_ev = int((qa_analysis_df["evasion_tier"] == "Intermediate (Hedging/Pivot)").sum())
    weighted_evasion_rate = round(((full_ev + 0.5 * inter_ev) / total_q) * 100, 1)

    qa_text = " ".join(qa_analysis_df["answer"].astype(str))
    n_words = max(1, len(qa_text.split()))
    qa_hedging_rate = round((len(HEDGING_LEXICON_RE.findall(qa_text)) / n_words) * 100, 2)
    qa_neg_pct = round(float((qa_analysis_df["answer_finbert_label"] == "negative").mean() * 100), 1)

    composite = round(fb_summary["FINBERT_SCORE"] - 0.35 * weighted_evasion_rate - 5.0 * qa_hedging_rate, 1)
    if fb_summary["FINBERT_SCORE"] >= 35.0 and weighted_evasion_rate < 30.0:
        signal, pos_size = "CONSTRUCTIVE", "High conviction"
    elif fb_summary["FINBERT_SCORE"] <= 15.0 or weighted_evasion_rate >= 45.0:
        signal, pos_size = "CAUTIOUS", "Risk-managed"
    else:
        signal, pos_size = "NEUTRAL", "Standard watch"

    return {
        **fb_summary,
        "ALPHA_SCORE": composite,
        "SIGNAL": signal,
        "POSITION_SIZE": pos_size,
        "Weighted_Evasion_Rate": weighted_evasion_rate,
        "Evasive_or_Hedged_Qs": f"{full_ev + inter_ev} / {total_q}",
        "QA_Hedging_Rate": qa_hedging_rate,
        "QA_Neg_Pct": qa_neg_pct,
    }

##Call Wrapper

In [12]:
def analyze_single_call(row: dict) -> Optional[dict]:
    turns_df = parse_transcript(row["transcript"], strip_safe_harbor=True)
    if turns_df.empty:
        return None
    qa_df = extract_qa_pairs(turns_df)
    if qa_df.empty:
        return None

    call_id = make_call_id(row)
    metrics_df = qa_df.apply(score_qa_exchange, axis=1)
    qa_analysis_df = pd.concat([qa_df, metrics_df], axis=1)
    qa_analysis_df["call_id"] = call_id

    turns_df["call_id"] = call_id
    chunks_df = create_turn_aware_chunks(turns_df, qa_df, call_id, row)
    fixed_chunks_df = create_fixed_chunks(turns_df, call_id, row)
    traj_df = score_finbert_qa_trajectory(chunks_df)

    fb_summary = compute_finbert_summary_fast(turns_df, qa_analysis_df)
    card = compute_call_card_metrics(fb_summary, qa_analysis_df)

    registry_entry = {
        "call_id": call_id,
        "ticker": str(row.get("ticker", "UNKNOWN")),
        "company": str(row.get("company", "UNKNOWN")),
        "quarter": str(row.get("datacqtr", "UNKNOWN")),
        "earnings_date": str(row.get("earnings_date", "UNKNOWN")),
        **card,
    }
    return {
        "registry": registry_entry,
        "turns": turns_df,
        "chunks": chunks_df,
        "fixed_chunks": fixed_chunks_df,
        "qa": qa_analysis_df,
        "traj": traj_df,
    }


def export_demo_batch(num_calls: int = NUM_DEMO_CALLS, seed: int = DEMO_SEED):
    registry, all_turns, all_chunks, all_fixed, all_qa, all_traj = [], [], [], [], [], []
    seen_tickers = set()
    t0 = time.time()

    for row in ds.shuffle(seed=seed):
        if len(registry) >= num_calls:
            break
        ticker = str(row.get("ticker", ""))
        if ticker in seen_tickers:
            continue
        res = analyze_single_call(row)
        if not res:
            continue
        seen_tickers.add(ticker)
        registry.append(res["registry"])
        all_turns.append(res["turns"])
        all_chunks.append(res["chunks"])
        all_fixed.append(res["fixed_chunks"])
        all_qa.append(res["qa"])
        all_traj.append(res["traj"])
        print(
            f"[{len(registry)}/{num_calls}] {res['registry']['ticker']} ({res['registry']['quarter']}) | "
            f"FinBERT={res['registry']['FINBERT_SCORE']:+.1f} | Evasion={res['registry']['Weighted_Evasion_Rate']}% | "
            f"QAs={len(res['qa'])}"
        )

    pd.concat(all_turns, ignore_index=True).to_parquet(DATA_DIR / "all_demo_turns.parquet", index=False)
    pd.concat(all_chunks, ignore_index=True).to_parquet(DATA_DIR / "all_demo_chunks.parquet", index=False)
    pd.concat(all_fixed, ignore_index=True).to_parquet(DATA_DIR / "all_demo_chunks_500w.parquet", index=False)
    pd.concat(all_qa, ignore_index=True).to_csv(DATA_DIR / "all_demo_qa_analysis.csv", index=False)
    pd.concat(all_traj, ignore_index=True).to_csv(DATA_DIR / "all_demo_finbert_trajectory.csv", index=False)
    (DATA_DIR / "demo_calls_registry.json").write_text(json.dumps(registry, indent=2), encoding="utf-8")

    print(f"\nExported {len(registry)} demo calls in {time.time() - t0:.1f}s to {DATA_DIR.resolve()}")
    return registry


demo_registry = export_demo_batch()

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[1/5] NSC (2015Q3) | FinBERT=+25.2 | Evasion=1.0% | QAs=48
[2/5] VLO (2017Q2) | FinBERT=+16.2 | Evasion=3.4% | QAs=29
[3/5] XYL (2015Q1) | FinBERT=+33.1 | Evasion=2.2% | QAs=23
[4/5] HD (2017Q3) | FinBERT=+32.8 | Evasion=1.5% | QAs=33
[5/5] ROL (2018Q2) | FinBERT=+35.6 | Evasion=0.0% | QAs=8

Exported 5 demo calls in 51.0s to /content/TA2_Earnings_Project_v7/data_processed


# 5. Phase 2 GPU — VRAM Cleanup, Hybrid + Rerank RAG & Qwen2.5-7B

##Explicit GPU VRAM Cleanup

In [13]:
for _var in ["nli_pipe", "finbert_pipe"]:
    if _var in globals():
        del globals()[_var]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU VRAM allocated after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

GPU VRAM allocated after cleanup: 2.28 GB


In [14]:
%%writefile rag_utils.py
import re
from typing import List, Optional
import numpy as np
from rank_bm25 import BM25Okapi

BM25_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "been", "being", "by", "can", "could",
    "did", "do", "does", "for", "from", "had", "has", "have", "he", "her", "him", "his",
    "how", "i", "if", "in", "into", "is", "it", "its", "me", "my", "of", "on", "or",
    "our", "out", "over", "said", "say", "she", "should", "so", "some", "than", "that",
    "the", "their", "them", "then", "there", "these", "they", "this", "those", "to",
    "too", "up", "us", "was", "we", "were", "what", "when", "where", "which", "who",
    "why", "will", "with", "would", "you", "your", "about", "also",
}


def tokenize_bm25(text: str) -> List[str]:
    return [
        tok for tok in re.findall(r"[a-z0-9%$]+", str(text).lower())
        if len(tok) >= 2 and tok not in BM25_STOPWORDS
    ]


def retrieve_top_k_chunks(
    query: str,
    call_id: str,
    collection,
    embedder,
    reranker=None,
    top_k: int = 3,
    candidate_k: int = 15,
    use_hybrid: bool = True,
    use_reranker: bool = True,
) -> List[dict]:
    """3-stage retrieval: Dense BGE-M3 -> Lexical BM25 Fusion -> Cross-Encoder Rerank."""
    q_vec = embedder.encode(query, normalize_embeddings=True).tolist()
    res = collection.query(
        query_embeddings=[q_vec],
        n_results=candidate_k,
        where={"call_id": call_id},
        include=["documents", "metadatas", "distances"],
    )
    docs = res["documents"][0] if res and res.get("documents") else []
    if not docs:
        return []

    ids = res["ids"][0]
    metas = res["metadatas"][0]
    dists = res["distances"][0]
    dense_scores = [max(0.0, 1.0 - float(d)) for d in dists]

    if use_hybrid and len(docs) > 1:
        tokenized_docs = [tokenize_bm25(d) for d in docs]
        bm25 = BM25Okapi(tokenized_docs)
        bm25_raw = bm25.get_scores(tokenize_bm25(query)).tolist()
        d_min, d_max = min(dense_scores), max(dense_scores)
        b_max = max(bm25_raw) or 1.0
        hybrid_scores = [
            0.65 * ((ds - d_min) / max(d_max - d_min, 1e-6)) + 0.35 * (bs / b_max)
            for ds, bs in zip(dense_scores, bm25_raw)
        ]
    else:
        bm25_raw = [0.0] * len(docs)
        hybrid_scores = list(dense_scores)

    candidates = [
        {
            "chunk_id": cid,
            "text": doc,
            "metadata": meta or {},
            "dense_score": round(ds, 4),
            "bm25_score": round(bs, 3),
            "hybrid_score": round(hs, 4),
            "rerank_score": round(hs, 4),
        }
        for cid, doc, meta, ds, bs, hs in zip(ids, docs, metas, dense_scores, bm25_raw, hybrid_scores)
    ]

    if use_reranker and reranker is not None:
        rr_scores = reranker.predict([(query, c["text"]) for c in candidates])
        for c, rs in zip(candidates, rr_scores):
            c["rerank_score"] = round(float(rs), 4)
        candidates.sort(key=lambda x: x["rerank_score"], reverse=True)
    else:
        candidates.sort(key=lambda x: x["hybrid_score"], reverse=True)

    return candidates[:top_k]

Writing rag_utils.py


##ChromaDB Indexing & Cross-Encoder Reranker

In [15]:
from rag_utils import retrieve_top_k_chunks

reranker = CrossEncoder(RERANKER_MODEL_ID, device=DEVICE_STR)
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))


def build_chroma_collection(name: str, df: pd.DataFrame):
    try:
        chroma_client.delete_collection(name)
    except Exception:
        pass
    col = chroma_client.create_collection(name=name, metadata={"hnsw:space": "cosine"})
    texts = df["text"].tolist()
    embs = embedder.encode(texts, batch_size=32, normalize_embeddings=True, show_progress_bar=False).tolist()
    metas = [{"call_id": str(r.call_id), "section": str(r.section), "ticker": str(r.ticker)} for r in df.itertuples(index=False)]
    col.add(ids=df["chunk_id"].tolist(), documents=texts, embeddings=embs, metadatas=metas)
    return col


demo_chunks_df = pd.read_parquet(DATA_DIR / "all_demo_chunks.parquet")
demo_fixed_df = pd.read_parquet(DATA_DIR / "all_demo_chunks_500w.parquet")

collection_turn_aware = build_chroma_collection("sp500_earnings_turn_aware", demo_chunks_df)
collection_500w = build_chroma_collection("sp500_earnings_chunks_500w", demo_fixed_df)
print(f"Indexed ChromaDB: Turn-Aware={collection_turn_aware.count()} chunks | 500w Baseline={collection_500w.count()} chunks")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Indexed ChromaDB: Turn-Aware=194 chunks | 500w Baseline=117 chunks


##Qwen2.5-7B-Instruct

In [17]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
) if torch.cuda.is_available() else None

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    device_map="auto" if torch.cuda.is_available() else None,
    quantization_config=bnb_config,
).eval()


def generate_qwen(prompt_val, max_new_tokens: int = 640) -> str:
    text = prompt_val.to_string() if hasattr(prompt_val, "to_string") else str(prompt_val)
    messages = [{"role": "user", "content": text}]
    formatted = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tokenizer(formatted, return_tensors="pt", truncation=True, max_length=6000).to(llm_model.device)
    with torch.inference_mode():
        out = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=llm_tokenizer.eos_token_id,
        )
    gen = out[0, inputs["input_ids"].shape[-1]:]
    return llm_tokenizer.decode(gen, skip_special_tokens=True).strip()


llm_runnable = RunnableLambda(lambda x: generate_qwen(x, max_new_tokens=640))
print(f"Loaded {LLM_MODEL_ID} cleanly. GPU VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB" if torch.cuda.is_available() else f"Loaded {LLM_MODEL_ID} on CPU.")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-7B-Instruct cleanly. GPU VRAM: 12.11 GB


# 6. Executive Summaries, Advanced ABSA Sentiment & Hybrid Reranked RAG

##Revenue Guidance, Highlights & Risk Factors Extraction

In [21]:
SUMMARY_PROMPT = ChatPromptTemplate.from_template(
    "You are an institutional equity research analyst. Read the executive Prepared Remarks for {company} ({quarter}) "
    "and return ONLY a valid JSON object with exactly these 4 keys:\n"
    "- \"revenue_guidance\": list of 2-4 concise bullet strings covering STRICTLY future-looking guidance, outlook, or targets.\n"
    "- \"key_highlights\": list of 3-4 concise bullet strings covering STRICTLY completed-quarter actual results (no overlap with guidance).\n"
    "- \"risk_factors\": list of 2-4 concise bullet strings covering headwinds, macro risks, tariffs, inflation, or operational challenges.\n"
    "- \"overall_outlook\": one of \"Positive\", \"Cautious/Mixed\", or \"Negative\".\n"
    "Preserve exact numbers and units from the text. Never invent figures.\n\n"
    "PREPARED REMARKS:\n{remarks}"
)
summary_chain = SUMMARY_PROMPT | llm_runnable | StrOutputParser()


def extract_json_from_text(raw: str) -> dict:
    raw = raw.strip()
    m = re.search(r"\{[\s\S]*\}", raw)
    if not m:
        return {}
    try:
        return json.loads(m.group(0))
    except Exception:
        return {}


# VRAM-safe: offload BGE-M3 to CPU while Qwen2.5-7B summarizes
if torch.cuda.is_available():
    embedder.to("cpu")
    torch.cuda.empty_cache()

turns_all_df = pd.read_parquet(DATA_DIR / "all_demo_turns.parquet")
summaries = {}

for entry in demo_registry:
    cid = entry["call_id"]
    prep_df = turns_all_df[(turns_all_df["call_id"] == cid) & (turns_all_df["section"] == "Prepared Remarks") & (turns_all_df["role"] == "Executive")]
    remarks = " ".join(" ".join(prep_df["text"].astype(str)).split()[:2000])
    raw_out = summary_chain.invoke({"company": entry["company"], "quarter": entry["quarter"], "remarks": remarks})
    parsed = extract_json_from_text(raw_out)
    if not parsed or "revenue_guidance" not in parsed:
        sents = sent_tokenize(remarks)
        parsed = {
            "revenue_guidance": [s for s in sents if re.search(r"guidance|expect|outlook|forecast", s, re.I)][:3] or ["No explicit numeric guidance stated."],
            "key_highlights": sents[:3],
            "risk_factors": [s for s in sents if re.search(r"headwind|risk|pressure|inflation|challenge", s, re.I)][:3] or ["No major operational risks highlighted."],
            "overall_outlook": "Cautious/Mixed",
        }
    summaries[cid] = parsed
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"Summary generated: {entry['ticker']} ({entry['quarter']}) -> Outlook: {parsed.get('overall_outlook')}")

(EVAL_DIR / "summaries.json").write_text(json.dumps(summaries, indent=2, ensure_ascii=False), encoding="utf-8")

if torch.cuda.is_available():
    embedder.to(DEVICE_STR)

Summary generated: NSC (2015Q3) -> Outlook: Cautious/Mixed
Summary generated: VLO (2017Q2) -> Outlook: Cautious/Mixed
Summary generated: XYL (2015Q1) -> Outlook: Cautious/Mixed
Summary generated: HD (2017Q3) -> Outlook: Positive
Summary generated: ROL (2018Q2) -> Outlook: Positive


##Aspect-Based Sentiment Analysis (ABSA)

In [22]:
class DriverSentiment(BaseModel):
    score: float = Field(..., ge=-1.0, le=1.0)
    rationale: str


class AdvancedCallSentiment(BaseModel):
    revenue_demand: DriverSentiment
    margins_costs: DriverSentiment
    capex_investment: DriverSentiment
    forward_guidance: DriverSentiment
    macro_tariff_risk: DriverSentiment
    management_confidence: float = Field(..., ge=0.0, le=10.0)
    hedging_level: Literal["Low", "Moderate", "High"]
    hedging_examples: List[str]
    prep_vs_qa_divergence: str


ABSA_PROMPT = ChatPromptTemplate.from_template(
    "Analyze the following earnings call excerpts for {company} ({quarter}). Return ONLY a valid JSON object matching this exact schema:\n"
    "{{\n"
    "  \"revenue_demand\": {{\"score\": -1.0 to 1.0, \"rationale\": \"1 sentence citing facts\"}},\n"
    "  \"margins_costs\": {{\"score\": -1.0 to 1.0, \"rationale\": \"1 sentence citing facts\"}},\n"
    "  \"capex_investment\": {{\"score\": -1.0 to 1.0, \"rationale\": \"1 sentence citing facts\"}},\n"
    "  \"forward_guidance\": {{\"score\": -1.0 to 1.0, \"rationale\": \"1 sentence citing facts\"}},\n"
    "  \"macro_tariff_risk\": {{\"score\": -1.0 to 1.0, \"rationale\": \"1 sentence citing facts\"}},\n"
    "  \"management_confidence\": 0.0 to 10.0,\n"
    "  \"hedging_level\": \"Low\" or \"Moderate\" or \"High\",\n"
    "  \"hedging_examples\": [\"exact short quote 1\", \"exact short quote 2\"],\n"
    "  \"prep_vs_qa_divergence\": \"1 sentence comparing scripted remarks vs unscripted Q&A tone\"\n"
    "}}\n\n"
    "PREPARED REMARKS EXCERPT:\n{prep_text}\n\n"
    "Q&A EXECUTIVE ANSWERS EXCERPT:\n{qa_text}"
)
absa_chain = ABSA_PROMPT | llm_runnable | StrOutputParser()

# VRAM-safe: offload BGE-M3 to CPU temporarily during 7B ABSA generation
if torch.cuda.is_available():
    embedder.to("cpu")
    torch.cuda.empty_cache()

qa_all_df = pd.read_csv(DATA_DIR / "all_demo_qa_analysis.csv")
absa_results = {}

for entry in demo_registry:
    cid = entry["call_id"]
    prep_text = " ".join(" ".join(turns_all_df[(turns_all_df["call_id"] == cid) & (turns_all_df["section"] == "Prepared Remarks") & (turns_all_df["role"] == "Executive")]["text"].astype(str)).split()[:900])
    qa_text = " ".join(" ".join(qa_all_df[qa_all_df["call_id"] == cid]["answer"].astype(str)).split()[:900])
    raw_absa = absa_chain.invoke({"company": entry["company"], "quarter": entry["quarter"], "prep_text": prep_text, "qa_text": qa_text})
    parsed = extract_json_from_text(raw_absa)
    try:
        validated = AdvancedCallSentiment(**parsed).model_dump()
    except Exception:
        validated = {
            "revenue_demand": {"score": 0.0, "rationale": "Fallback neutral assessment."},
            "margins_costs": {"score": 0.0, "rationale": "Fallback neutral assessment."},
            "capex_investment": {"score": 0.0, "rationale": "Fallback neutral assessment."},
            "forward_guidance": {"score": 0.0, "rationale": "Fallback neutral assessment."},
            "macro_tariff_risk": {"score": 0.0, "rationale": "Fallback neutral assessment."},
            "management_confidence": 5.0,
            "hedging_level": "Moderate",
            "hedging_examples": [],
            "prep_vs_qa_divergence": "Scripted remarks and Q&A tone appeared broadly consistent.",
        }
    absa_results[cid] = validated
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"ABSA validated: {entry['ticker']} | Confidence={validated['management_confidence']}/10 | Hedging={validated['hedging_level']}")

(EVAL_DIR / "absa_sentiment.json").write_text(json.dumps(absa_results, indent=2, ensure_ascii=False), encoding="utf-8")

if torch.cuda.is_available():
    embedder.to(DEVICE_STR)

ABSA validated: NSC | Confidence=7.0/10 | Hedging=Moderate
ABSA validated: VLO | Confidence=7.0/10 | Hedging=Moderate
ABSA validated: XYL | Confidence=7.0/10 | Hedging=Moderate
ABSA validated: HD | Confidence=8.0/10 | Hedging=Low
ABSA validated: ROL | Confidence=8.0/10 | Hedging=Low


##Hybrid + Cross-Encoder Reranked RAG QA Demo

In [23]:
RAG_PROMPT = ChatPromptTemplate.from_template(
    "You are a financial analyst. Answer the question using ONLY the 3 retrieved transcript excerpts below. "
    "Cite sources as [1], [2], [3]. If the information is not present, state that clearly.\n\n"
    "EXCERPTS:\n{context}\n\nQUESTION: {question}"
)
rag_chain = RAG_PROMPT | llm_runnable | StrOutputParser()


def answer_question_rag(question: str, call_id: str, top_k: int = 3) -> dict:
    hits = retrieve_top_k_chunks(
        question, call_id, collection_turn_aware, embedder, reranker=reranker, top_k=top_k
    )
    context = "\n\n".join(f"[{i}] ({h['metadata'].get('section', 'Transcript')}): {h['text']}" for i, h in enumerate(hits, 1))
    answer = rag_chain.invoke({"context": context, "question": question})
    return {"question": question, "answer": answer, "hits": hits}


sample_call = demo_registry[0]
rag_demo = answer_question_rag("What did management say about revenue growth, margins, or capital allocation?", sample_call["call_id"])
print(f"=== RAG Demo: {sample_call['ticker']} ({sample_call['quarter']}) ===")
print(rag_demo["answer"])

=== RAG Demo: NSC (2015Q3) ===
Management indicated several points regarding revenue growth, margins, and capital allocation:

1. **Revenue Growth**: Management stated that they aim to grow the top lines through pricing based on market conditions at a rate better than rail inflation. They also mentioned that revenue growth excluding fuel surcharge impacts was positive in the second quarter and expected to continue for the remainder of the year [3].

2. **Margins**: While specific mentions of margins were not directly addressed in the provided excerpts, the discussion around revenue growth and pricing suggests an effort to improve profitability. Additionally, management noted the need to prioritize capital spending in favor of revenue growth opportunities and profit margin [3].

3. **Capital Allocation**: Management discussed a reduction in capital expenditures (CapEx) for the current year, which was a 5% reduction from the previous year. They also mentioned a target to manage capital s

# 7. Summary Factuality Check, ROUGE & 3-Stage RAG Ablation Study

##Summary Factuality (Numeric & Semantic Grounding) + ROUGE

In [24]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
NUM_RE = re.compile(r"\$?\d+(?:[\.,]\d+)?%?")

eval_rows = []
for entry in demo_registry:
    cid = entry["call_id"]
    summ = summaries.get(cid, {})
    bullets = (
        summ.get("revenue_guidance", [])
        + summ.get("key_highlights", [])
        + summ.get("risk_factors", [])
    )
    summary_text = " ".join(str(b) for b in bullets)
    source_text = " ".join(
        turns_all_df[(turns_all_df["call_id"] == cid) & (turns_all_df["section"] == "Prepared Remarks") & (turns_all_df["role"] == "Executive")]["text"].astype(str)
    )

    # 1. Numeric grounding
    summ_nums = [re.sub(r"[\$,%]", "", n) for n in NUM_RE.findall(summary_text)]
    src_nums = set(re.sub(r"[\$,%]", "", n) for n in NUM_RE.findall(source_text))
    num_grounding = (sum(1 for n in summ_nums if n in src_nums) / len(summ_nums) * 100) if summ_nums else 100.0

    # 2. Semantic grounding (BGE-M3 cosine similarity)
    src_sents = [s for s in sent_tokenize(source_text) if len(s.split()) >= 5][:200]
    if bullets and src_sents:
        b_embs = embedder.encode(bullets, convert_to_tensor=True, normalize_embeddings=True)
        s_embs = embedder.encode(src_sents, convert_to_tensor=True, normalize_embeddings=True)
        sem_grounding = float(util.cos_sim(b_embs, s_embs).max(dim=1).values.mean())
    else:
        sem_grounding = 0.0

    # 3. ROUGE vs extractive reference
    ref_sents = [s for s in src_sents if re.search(r"revenue|margin|guidance|expect|growth|risk|headwind", s, re.I)][:10]
    ref_text = " ".join(ref_sents) if ref_sents else " ".join(src_sents[:10])
    r_scores = scorer.score(ref_text, summary_text)

    eval_rows.append({
        "ticker": entry["ticker"],
        "quarter": entry["quarter"],
        "numeric_grounding_pct": round(num_grounding, 1),
        "semantic_grounding": round(sem_grounding, 3),
        "rouge1_f1": round(r_scores["rouge1"].fmeasure, 3),
        "rouge2_f1": round(r_scores["rouge2"].fmeasure, 3),
        "rougeL_f1": round(r_scores["rougeL"].fmeasure, 3),
    })

summary_eval_df = pd.DataFrame(eval_rows)
summary_eval_df.to_csv(OUT_DIR / "summary_evaluation.csv", index=False)
display(summary_eval_df)

,ticker,quarter,numeric_grounding_pct,semantic_grounding,rouge1_f1,rouge2_f1,rougeL_f1
0,NSC,2015Q3,100.0,0.888,0.417,0.164,0.215
1,VLO,2017Q2,100.0,0.959,0.392,0.147,0.146
2,XYL,2015Q1,100.0,0.788,0.411,0.157,0.156
3,HD,2017Q3,100.0,0.906,0.572,0.377,0.263
4,ROL,2018Q2,100.0,0.776,0.410,0.152,0.227


In [25]:
# Cell 7.2 — 3-Stage RAG Ablation Study (500w Dense vs Turn-Aware Dense vs Hybrid + Rerank)
ablation_qs = []
for entry in demo_registry:
    cid = entry["call_id"]
    sub_qa = qa_all_df[qa_all_df["call_id"] == cid].head(4)
    for r in sub_qa.itertuples(index=False):
        gold_snippet = " ".join(str(r.answer).split()[:18]).lower()
        ablation_qs.append({"call_id": cid, "question": str(r.question), "gold_snippet": gold_snippet})


def eval_rag_config(name: str, col, use_hybrid: bool, use_reranker: bool) -> dict:
    top1, top3, mrr = 0, 0, 0.0
    for item in ablation_qs:
        hits = retrieve_top_k_chunks(
            item["question"], item["call_id"], col, embedder,
            reranker=reranker if use_reranker else None,
            top_k=3, use_hybrid=use_hybrid, use_reranker=use_reranker,
        )
        rank = None
        for idx, h in enumerate(hits, 1):
            if item["gold_snippet"] in h["text"].lower():
                rank = idx
                break
        if rank == 1:
            top1 += 1
        if rank is not None and rank <= 3:
            top3 += 1
            mrr += 1.0 / rank

    n = max(1, len(ablation_qs))
    return {
        "Configuration": name,
        "Eval Queries (n)": n,
        "Top-1 Hit Rate (%)": round(top1 / n * 100, 1),
        "Top-3 Hit Rate (%)": round(top3 / n * 100, 1),
        "MRR@3": round(mrr / n, 3),
    }


rag_ablation_df = pd.DataFrame([
    eval_rag_config("1. Baseline: 500w Fixed Chunks (Dense Only)", collection_500w, False, False),
    eval_rag_config("2. Turn-Aware / Q&A-Dyad (Dense Only)", collection_turn_aware, False, False),
    eval_rag_config("3. Optimal v7: Turn-Aware (Hybrid BM25 + Rerank)", collection_turn_aware, True, True),
])
rag_ablation_df.to_csv(OUT_DIR / "rag_ablation_comparison.csv", index=False)
display(rag_ablation_df)

,Configuration,Eval Queries (n),Top-1 Hit Rate (%),Top-3 Hit Rate (%),MRR@3
0,1. Baseline: 500w Fixed Chunks (Dense Only),20,90.0,100.0,0.942
1,2. Turn-Aware / Q&A-Dyad (Dense Only),20,95.0,100.0,0.975
2,3. Optimal v7: Turn-Aware (Hybrid BM25 + Rerank),20,95.0,100.0,0.975


# 8. Executive Dashboard & Live Demo

##Visual Prototyping

In [26]:
def plot_call_tone_and_evasion(call_id: str):
    traj = pd.read_csv(DATA_DIR / "all_demo_finbert_trajectory.csv")
    qa = pd.read_csv(DATA_DIR / "all_demo_qa_analysis.csv")
    t_sub = traj[traj["call_id"] == call_id].sort_values("section_chunk_idx")
    q_sub = qa[qa["call_id"] == call_id].sort_values("pair_id")

    fig = make_subplots(rows=2, cols=1, subplot_titles=("FinBERT Q&A Tone Trajectory", "Management Evasion Score per Question"), vertical_spacing=0.18)
    fig.add_trace(go.Scatter(x=t_sub["section_chunk_idx"] + 1, y=t_sub["finbert_net_score"], mode="lines+markers", line=dict(color="#42669a", width=3)), row=1, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=1)

    colors = {"Direct": "#39776e", "Intermediate (Hedging/Pivot)": "#a96d22", "Fully Evasive": "#995169"}
    fig.add_trace(go.Bar(x=[f"Q{p}" for p in q_sub["pair_id"]], y=q_sub["evasion_score"], marker_color=[colors.get(t, "#42669a") for t in q_sub["evasion_tier"]]), row=2, col=1)
    fig.add_hline(y=62, line_dash="dot", line_color="#995169", row=2, col=1)
    fig.update_layout(height=580, template="plotly_white", showlegend=False)
    fig.write_html(str(OUT_DIR / f"{call_id}_preview.html"))
    fig.show()


plot_call_tone_and_evasion(demo_registry[0]["call_id"])

In [27]:
%%writefile app.py
import json
import math
import os
import re
import urllib.request
from collections import Counter
from html import escape
from pathlib import Path

import chromadb
import pandas as pd
import plotly.graph_objects as go
import streamlit as st
from sentence_transformers import SentenceTransformer

st.set_page_config(
    page_title="Earnings Call Review",
    layout="wide",
    initial_sidebar_state="expanded",
)

BASE_DIR = Path(os.environ.get("EARNINGS_PROJECT_DIR", "./TA2_Earnings_Project_v7"))
DATA_DIR = BASE_DIR / "data_processed"
CHROMA_DIR = BASE_DIR / "chroma_db"
EVAL_DIR = BASE_DIR / "eval_data"
OUT_DIR = BASE_DIR / "final_outputs"

BLUE = "#42669a"
TEAL = "#39776e"
AMBER = "#a96d22"
BERRY = "#995169"
INK = "#22334b"

OFFLINE = "Offline Grounded RAG (Local Qwen)"
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
GEMINI = f"Google Gemini ({GEMINI_MODEL})"
OPENAI = "OpenAI (gpt-4o-mini)"

STOPWORDS = set("""
a an and are as at be been being by can could did do does for from had has have
he her him his how i if in into is it its me my no not of on or our out over
said say she should so some than that the their them then there these they this
those to too up us was we were what when where which who why will with would
you your about also
""".split())


def tokenize(text):
    return [
        token for token in re.findall(r"[a-z0-9%$]+", str(text).lower())
        if len(token) >= 2 and token not in STOPWORDS
    ]


def bm25_scores(question, documents):
    query = tokenize(question)
    docs = [tokenize(text) for text in documents]
    average_length = max(1, sum(map(len, docs)) / len(docs))
    document_frequency = Counter()
    for tokens in docs:
        document_frequency.update(set(tokens))

    scores = []
    for tokens in docs:
        frequency = Counter(tokens)
        length_penalty = 1.5 * (0.25 + 0.75 * len(tokens) / average_length)
        score = 0.0
        for term in query:
            count = frequency[term]
            if count:
                df = document_frequency[term]
                weight = math.log1p((len(docs) - df + 0.5) / (df + 0.5))
                score += weight * count * 2.5 / (count + length_penalty)
        scores.append(score)
    return scores


@st.cache_data
def load_dashboard_data():
    registry_file = DATA_DIR / "demo_calls_registry.json"
    summary_file = EVAL_DIR / "summaries.json"

    def read_csv(path):
        return pd.read_csv(path) if path.is_file() else pd.DataFrame()

    return (
        json.loads(registry_file.read_text(encoding="utf-8")) if registry_file.is_file() else [],
        json.loads(summary_file.read_text(encoding="utf-8")) if summary_file.is_file() else {},
        read_csv(DATA_DIR / "all_demo_qa_analysis.csv"),
        read_csv(DATA_DIR / "all_demo_finbert_trajectory.csv"),
        read_csv(OUT_DIR / "baseline_vs_custom_comparison.csv"),
    )


@st.cache_resource
def rag_embedder():
    return SentenceTransformer("BAAI/bge-m3")


@st.cache_resource
def rag_collection():
    return chromadb.PersistentClient(path=str(CHROMA_DIR)).get_collection(
        "sp500_earnings_turn_aware"
    )


@st.cache_data(show_spinner=False)
def exported_passages(path, modified, call_id):
    frame = pd.read_parquet(
        path,
        filters=[("call_id", "==", call_id)],
        columns=["chunk_id", "text", "section", "call_id"],
    )
    return [
        {
            "chunk_id": row.chunk_id,
            "text": row.text,
            "metadata": {"section": row.section, "call_id": row.call_id},
        }
        for row in frame.itertuples(index=False)
        if isinstance(row.text, str) and row.text.strip()
    ]


@st.cache_resource(show_spinner=False, max_entries=12)
def passage_vectors(texts):
    return rag_embedder().encode(
        list(texts), normalize_embeddings=True, show_progress_bar=False
    )


def retrieve_hybrid_chunks(question, call_id, limit=8):
    export = DATA_DIR / "all_demo_chunks.parquet"
    hits = {}

    if (CHROMA_DIR / "chroma.sqlite3").is_file():
        try:
            rows = rag_collection().get(
                where={"call_id": call_id},
                include=["documents", "metadatas"],
            )
            for chunk_id, text, metadata in zip(
                rows["ids"], rows["documents"], rows["metadatas"]
            ):
                if text and text.strip():
                    hits[chunk_id] = dict(
                        chunk_id=chunk_id, text=text, metadata=metadata or {}
                    )
        except Exception:
            if not export.is_file():
                raise

    if export.is_file():
        for hit in exported_passages(str(export), export.stat().st_mtime_ns, call_id):
            hits.setdefault(hit["chunk_id"], hit)

    if not hits:
        return []

    passages = list(hits.values())
    texts = tuple(hit["text"] for hit in passages)
    query_vector = rag_embedder().encode(question, normalize_embeddings=True)
    dense = (passage_vectors(texts) @ query_vector).clip(0, 1).tolist()
    lexical = bm25_scores(question, texts)
    low, high = min(dense), max(dense)
    lexical_max = max(lexical) or 1.0

    ranked = []
    for hit, similarity, bm25 in zip(passages, dense, lexical):
        score = 0.70 * (similarity - low) / max(high - low, 1e-6)
        score += 0.30 * bm25 / lexical_max
        ranked.append(dict(hit, score=score, dense_score=similarity, bm25_score=bm25))
    return sorted(ranked, key=lambda hit: hit["score"], reverse=True)[:limit]


@st.cache_resource
def load_rag_qwen():
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    if torch.cuda.is_available():
        from transformers import BitsAndBytesConfig

        model_id = "Qwen/Qwen2.5-7B-Instruct"
        options = {
            "device_map": "auto",
            "quantization_config": BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
            ),
        }
    else:
        model_id = "Qwen/Qwen2.5-0.5B-Instruct"
        options = {}

    tokenizer = AutoTokenizer.from_pretrained(model_id, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, local_files_only=True, **options
    ).eval()
    return tokenizer, model


def post_json(url, headers, payload):
    request = urllib.request.Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json", **headers},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=60) as response:
        return json.load(response)


def ask_model(instructions, data, provider, api_key, json_output=False):
    content = json.dumps(data, ensure_ascii=False)
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": content},
    ]

    if provider == OFFLINE:
        import torch

        tokenizer, model = load_rag_qwen()
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.inference_mode():
            output = model.generate(
                **inputs,
                max_new_tokens=360,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated = output[0, inputs["input_ids"].shape[-1]:]
        return tokenizer.decode(generated, skip_special_tokens=True).strip()

    if provider == GEMINI:
        payload = {
            "systemInstruction": {"parts": [{"text": instructions}]},
            "contents": [{"role": "user", "parts": [{"text": content}]}],
        }
        if json_output:
            payload["generationConfig"] = {"responseMimeType": "application/json"}
        response = post_json(
            f"https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent",
            {"x-goog-api-key": api_key},
            payload,
        )
        parts = response["candidates"][0]["content"]["parts"]
        return "".join(p.get("text", "") for p in parts if not p.get("thought")).strip()

    payload = {"model": "gpt-4o-mini", "messages": messages}
    if json_output:
        payload["response_format"] = {"type": "json_object"}
    response = post_json(
        "https://api.openai.com/v1/chat/completions",
        {"Authorization": f"Bearer {api_key}"},
        payload,
    )
    return (response["choices"][0]["message"].get("content") or "").strip()


def read_json_object(raw):
    raw = raw.strip()
    if raw.startswith("```") and raw.endswith("```"):
        raw = "\n".join(raw.splitlines()[1:-1])
    try:
        result = json.loads(raw)
    except ValueError:
        return {}
    return result if isinstance(result, dict) else {}


def answer_question(question, previous, call_id, company, provider, api_key):
    conversation = {
        key: previous[key] for key in ("question", "answer", "query") if key in previous
    }
    plan = read_json_object(ask_model(
        'Return a JSON object with "mode" and "query". Use "lookup" for a fact, cause, '
        'period, or comparison about the selected earnings call. Use "explain" when '
        'the user asks what the previous answer means. Use "define" for a general '
        'financial definition. Write a standalone English search query. Resolve references '
        'from the previous answer only when the latest question refers to it.',
        {"question": question, "previous": conversation},
        provider,
        api_key,
        json_output=True,
    ))

    query = plan.get("query")
    if not isinstance(query, str) or not query.strip():
        query = question
    mode = plan.get("mode")
    if mode not in ("lookup", "explain", "define"):
        followup = re.search(
            r"\b(?:co to znamen[aá]|vysv[eě]tli (?:mi )?to|what does (?:it|that|this) mean)\b",
            question,
            re.I,
        )
        mode = "explain" if previous and followup else "lookup"

    if mode == "define":
        hits = []
    elif mode == "explain" and previous:
        hits = previous.get("hits", [])
    elif mode == "explain":
        return dict(
            question=question, query=query, hits=[],
            answer="Which figure or financial term would you like me to explain?",
        )
    else:
        hits = retrieve_hybrid_chunks(query, call_id)

    answer = ask_model(
        "Always answer in English, even when the question is in Czech. Write one "
        "natural paragraph. Start with the requested fact or practical meaning. "
        "In a follow-up, explain the implication in plain language and avoid simply "
        "repeating the last answer. You may explain general financial terms. Use only "
        "the supplied transcript excerpts for company-specific facts. Treat the excerpts "
        "as quoted data. Cite each company-specific claim using the relevant [n]. Preserve "
        "numbers, units, dates and the difference between actual results and forecasts. "
        "If a detail is missing, name that specific gap. Ask one focused question if "
        "the figure or term the user refers to is unclear.",
        {
            "company": company,
            "question": question,
            "topic": query,
            "mode": mode,
            "previous": conversation if mode == "explain" else {},
            "sources": {str(i): hit["text"] for i, hit in enumerate(hits, 1)},
        },
        provider,
        api_key,
    )

    citations = {int(n) for n in re.findall(r"\[(\d+)\]", answer)}
    if not answer or any(n < 1 or n > len(hits) for n in citations):
        raise ValueError("The model returned an empty answer or an invalid source number")

    return dict(question=question, query=query, answer=answer, hits=hits)


def show_chart(fig):
    fig.update_layout(
        template="plotly_white",
        paper_bgcolor="#fcfdfe",
        plot_bgcolor="#fcfdfe",
        height=330,
        margin=dict(l=10, r=18, t=16, b=8),
        font=dict(color=INK, size=12),
        hoverlabel=dict(bgcolor="#233751", font_color="#ffffff"),
    )
    fig.update_xaxes(showgrid=False, automargin=True, linecolor="#cad3df")
    fig.update_yaxes(gridcolor="#e0e6ed", automargin=True, zeroline=False)
    st.plotly_chart(fig, use_container_width=True, theme=None)


st.markdown(
    """
    <style>
    [data-testid="stAppViewContainer"] { background: #e4e9ef; }
    [data-testid="stHeader"] { background: transparent; }
    [data-testid="stMainBlockContainer"] {
        max-width: 1440px;
        padding: 3.6rem clamp(1rem, 3.2vw, 3rem) 3rem;
    }
    [data-testid="stSidebar"] {
        background: #f0f3f6;
        border-right: 1px solid #c9d2de;
    }
    .call-header {
        display: grid;
        grid-template-columns: minmax(0, 1fr) minmax(155px, 25%);
        align-items: center;
        gap: 1.6rem;
        background: #283c56;
        border-radius: 15px;
        padding: clamp(1.4rem, 3.4vw, 2.5rem);
        margin-bottom: .5rem;
    }
    .call-header .ticker { color: #c0d0e2; font-weight: 700; }
    .call-header h1 {
        color: #fff;
        font: 500 clamp(2rem, 4vw, 3.5rem)/1.06 Georgia, serif;
        margin: 0;
        padding: 0;
        overflow-wrap: anywhere;
    }
    .call-header .date, .call-header .size { color: #cbd6e2; margin: .65rem 0 0; }
    .call-header .signal { border-left: 1px solid #67798e; padding-left: 1.25rem; }
    .call-header .signal-label { color: #ecd4a6; font-weight: 650; font-size: .77rem; }
    .call-header .signal-value { color: #fff; font-size: 1.8rem; font-weight: 650; }
    [data-testid="stMetric"], [data-testid="stPlotlyChart"] {
        background: #fcfdfe;
        border: 1px solid #cbd4df;
        border-radius: 12px;
        padding: .8rem;
    }
    [data-testid="stMetricLabel"] p { color: #526278; white-space: normal; }
    [data-testid="stMetricValue"] { color: #243951; }
    [data-testid="stTabs"] [role="tablist"] {
        gap: clamp(.7rem, 2vw, 1.8rem);
        border-bottom: 1px solid #c0cad7;
    }
    [data-testid="stTabs"] [aria-selected="true"] p { color: #2c5183; font-weight: 700; }
    @media (max-width: 700px) {
        .call-header { grid-template-columns: 1fr; gap: 1rem; }
        .call-header .signal { border-left: 0; border-top: 1px solid #67798e; padding: .8rem 0 0; }
    }
    </style>
    """,
    unsafe_allow_html=True,
)

registry, summaries, qa_df, traj_df, comp_df = load_dashboard_data()
if not registry:
    st.error(f"No call registry found in {DATA_DIR.resolve()}. Run the analytical notebook sections first.")
    st.stop()

calls = {row["call_id"]: row for row in registry}


def call_label(call_id):
    row = calls[call_id]
    return f"{row['ticker']} · {row['quarter']} · {row['company']}"


st.sidebar.header("Earnings Call Review")
cid = st.sidebar.selectbox("Earnings call", list(calls), format_func=call_label)
ticket = calls[cid]

other_ids = [call_id for call_id in calls if call_id != cid]
reference = None
if other_ids:
    compare_id = st.sidebar.selectbox(
        "Compare with",
        [None, *other_ids],
        format_func=lambda value: "No comparison" if value is None else call_label(value),
        key=f"compare_{cid}",
    )
    if compare_id is not None:
        reference = calls[compare_id]

st.sidebar.divider()
st.sidebar.subheader("Transcript Q&A")
provider = st.sidebar.selectbox("Answer mode", [OFFLINE, GEMINI, OPENAI])
api_key = ""
if provider != OFFLINE:
    api_key = st.sidebar.text_input("API key", type="password", key=provider).strip()


def header_value(field):
    return escape(str(ticket[field]))


st.markdown(
    f"""
    <div class="call-header">
      <div>
        <p class="ticker">{header_value("ticker")} &nbsp;·&nbsp; {header_value("quarter")}</p>
        <h1>{header_value("company")}</h1>
        <p class="date">Earnings call &nbsp;·&nbsp; {header_value("earnings_date")}</p>
      </div>
      <div class="signal">
        <div class="signal-label">CALL ASSESSMENT</div>
        <div class="signal-value">{header_value("SIGNAL")}</div>
        <p class="size">Stance: {header_value("POSITION_SIZE")}</p>
      </div>
    </div>
    """,
    unsafe_allow_html=True,
)

numeric_metrics = [
    ("Composite score", "ALPHA_SCORE", "+.1f", "", " pts"),
    ("FinBERT score", "FINBERT_SCORE", "+.1f", "", " pts"),
    ("Weighted evasion", "Weighted_Evasion_Rate", ".1f", "%", " pp"),
]

if reference is not None:
    st.caption(f"KPI differences below are relative to {call_label(reference['call_id'])}.")

columns = st.columns(4)
comparison_rows = []
for column, (label, field, fmt, suffix, unit) in zip(columns[:3], numeric_metrics):
    value = float(ticket[field])
    current_text = f"{value:{fmt}}{suffix}"
    delta = None
    if reference is not None:
        baseline = float(reference[field])
        delta = f"{value - baseline:+.1f}{unit}"
        comparison_rows.append({
            "Metric": label,
            "Selected call": current_text,
            "Comparison call": f"{baseline:{fmt}}{suffix}",
            "Difference": delta,
        })
    elif field == "FINBERT_SCORE":
        delta = f"{ticket['FINBERT_TONE_DELTA']:+.1f} Q&A shift"
    column.metric(label, current_text, delta=delta, delta_color="off")

columns[3].metric("Evasive / hedged questions", str(ticket["Evasive_or_Hedged_Qs"]))

if reference is not None:
    for label, field in [
        ("Evasive / hedged questions", "Evasive_or_Hedged_Qs"),
        ("Call assessment", "SIGNAL"),
        ("Stance", "POSITION_SIZE"),
    ]:
        comparison_rows.append({
            "Metric": label,
            "Selected call": str(ticket[field]),
            "Comparison call": str(reference[field]),
            "Difference": "—",
        })
    with st.expander("Compare call details"):
        st.dataframe(pd.DataFrame(comparison_rows), hide_index=True, use_container_width=True)

summary_tab, qa_tab, rag_tab, eval_tab = st.tabs([
    "Executive summary",
    "Q&A explorer",
    "Ask the transcript",
    "Evaluation & diagnostics",
])

with summary_tab:
    summary = summaries.get(cid)
    if summary:
        st.subheader(f"Prepared remarks · {summary.get('overall_outlook', 'N/A')}")
        left, right = st.columns(2)
        with left, st.container(border=True):
            st.markdown("#### Future revenue & financial guidance")
            for item in summary.get("revenue_guidance", []):
                st.markdown(f"- {item}")
            st.markdown("#### Completed-quarter highlights")
            for item in summary.get("key_highlights", []):
                st.markdown(f"- {item}")
        with right, st.container(border=True):
            st.markdown("#### Risk factors & headwinds")
            for item in summary.get("risk_factors", []):
                st.markdown(f"- {item}")
            st.markdown("#### Section tone · FinBERT")
            st.write(f"**Prepared remarks:** {ticket['FINBERT_PREP_SCORE']:+.1f}")
            st.write(f"**Q&A section:** {ticket['FINBERT_QA_SCORE']:+.1f}")
            st.write(f"**Q&A minus prepared remarks:** {ticket['FINBERT_TONE_DELTA']:+.1f}")
    else:
        st.info("Run section 6.1 to generate the summary for this call.")

with qa_tab:
    trajectory = (
        traj_df[traj_df["call_id"] == cid].sort_values("section_chunk_idx")
        if not traj_df.empty else pd.DataFrame()
    )
    exchanges = (
        qa_df[qa_df["call_id"] == cid].sort_values("pair_id")
        if not qa_df.empty else pd.DataFrame()
    )
    st.subheader("FinBERT tone through the call")
    if trajectory.empty:
        st.info("No Q&A tone trajectory is available for this call.")
    else:
        tone_fig = go.Figure(go.Scatter(
            x=trajectory["section_chunk_idx"] + 1,
            y=trajectory["finbert_net_score"],
            mode="lines+markers",
            line=dict(color=BLUE, width=3),
            marker=dict(size=7),
            hovertemplate="Chunk %{x}<br>FinBERT %{y:.1f}<extra></extra>",
        ))
        tone_fig.add_hline(y=0, line_dash="dash", line_color="#8895a5")
        tone_fig.update_layout(xaxis_title="Q&A chunk", yaxis_title="FinBERT net score")
        show_chart(tone_fig)

    st.subheader("Analyst questions")
    if exchanges.empty:
        st.info("No analyst exchanges are available for this call.")
    else:
        tiers = exchanges["evasion_tier"].dropna().unique().tolist()
        filter_col, score_col = st.columns(2)
        selected_tiers = filter_col.multiselect(
            "Evasion tier", tiers, default=tiers, key=f"tiers_{cid}"
        )
        low, high = score_col.slider(
            "Evasion score range", 0, 100, (0, 100), key=f"range_{cid}"
        )
        scores = pd.to_numeric(exchanges["evasion_score"], errors="coerce")
        filtered = exchanges[
            exchanges["evasion_tier"].isin(selected_tiers) & scores.between(low, high)
        ].reset_index(drop=True)
        st.caption(f"Showing {len(filtered)} of {len(exchanges)} questions.")

        if filtered.empty:
            st.info("Widen the score range or select another evasion tier.")
        else:
            colors = {
                "Direct": TEAL,
                "Intermediate (Hedging/Pivot)": AMBER,
                "Fully Evasive": BERRY,
            }
            bar_fig = go.Figure(go.Bar(
                x=[f"Q{pair}" for pair in filtered["pair_id"]],
                y=filtered["evasion_score"],
                marker_color=[colors.get(tier, BLUE) for tier in filtered["evasion_tier"]],
                customdata=filtered["evasion_tier"],
                hovertemplate="%{x}<br>%{customdata}<br>Score %{y}/100<extra></extra>",
            ))
            bar_fig.add_hline(y=62, line_dash="dot", line_color=BERRY)
            bar_fig.update_layout(
                xaxis_title="Analyst question",
                yaxis_title="Evasion score",
                yaxis_range=[0, 105],
                bargap=0.32,
            )
            show_chart(bar_fig)
            with st.container(border=True):
                row_index = st.selectbox(
                    "Read a question and answer",
                    list(filtered.index),
                    format_func=lambda i: (
                        f"Q{filtered.at[i, 'pair_id']} · {filtered.at[i, 'analyst']}"
                        f" · {filtered.at[i, 'evasion_tier']}"
                        f" ({filtered.at[i, 'evasion_score']}/100)"
                    ),
                    key=f"exchange_{cid}",
                )
                row = filtered.loc[row_index]
                st.markdown(f"**Analyst · {row['analyst']}**")
                st.write(row["question"])
                st.markdown(f"**Management · {row['executives']}**")
                st.write(row["answer"])

            with st.expander("View filtered exchange table"):
                display_cols = [
                    c for c in [
                        "pair_id", "analyst", "executives", "evasion_tier",
                        "evasion_score", "semantic_alignment", "answer_finbert_label",
                    ] if c in filtered.columns
                ]
                st.dataframe(
                    filtered[display_cols],
                    hide_index=True,
                    use_container_width=True,
                )

with rag_tab:
    st.subheader(f"Ask about {ticket['ticker']} · {ticket['quarter']}")
    st.caption("Ask a follow-up in the same box. Start with 'New:' to reset the context.")

    with st.form(f"rag_form_{cid}", clear_on_submit=True):
        question = st.text_input(
            "Question about this transcript",
            placeholder="What did management say about margins or capital expenditure?",
        )
        submitted = st.form_submit_button("Search & answer", type="primary")

    answer_key = f"rag_answer_{cid}_{provider}"
    saved = st.session_state.get(answer_key, {})

    if submitted and question.strip():
        current = question.strip()
        previous = saved
        if current.lower().startswith("new:"):
            current = current[4:].strip()
            previous = {}
            st.session_state.pop(answer_key, None)
            saved = {}

        if current:
            if provider != OFFLINE and not api_key:
                st.error("Enter an API key in the sidebar.")
            else:
                try:
                    with st.spinner("Reading the transcript…"):
                        saved = answer_question(
                            current, previous, cid, ticket["company"], provider, api_key
                        )
                    st.session_state[answer_key] = saved
                except OSError as exc:
                    st.error(f"Could not run {provider}. Verify that the local models are cached or check the API connection.")
                    with st.expander("Error details"):
                        st.code(str(exc))
                except Exception as exc:
                    st.error(f"Could not generate an answer with {provider} ({type(exc).__name__}).")

    if saved:
        st.markdown("#### Answer")
        st.caption(f"{saved['question']} · {provider}")
        st.markdown(f"<p>{escape(saved['answer'])}</p>", unsafe_allow_html=True)

        if saved["hits"]:
            st.markdown("#### Retrieved sources")
        for number, hit in enumerate(saved["hits"], 1):
            section = hit["metadata"].get("section", "Transcript")
            with st.expander(f"[{number}] {hit['chunk_id']} · {section} · hybrid {hit['score']:.3f}"):
                st.caption(f"Dense {hit['dense_score']:.3f} · BM25 {hit['bm25_score']:.2f}")
                st.write(hit["text"])

with eval_tab:
    st.subheader("Pipeline evaluation & diagnostics")

    for filename, title in [
        ("rag_ablation_comparison.csv", "RAG ablation · dense and hybrid + rerank retrieval"),
        ("summary_evaluation.csv", "Summary factuality & ROUGE"),
        ("oos_strategy_scorecard.csv", "Out-of-sample strategy scorecard (from Backtest Colab)"),
        ("strategy_scorecard.csv", "Out-of-sample strategy scorecard"),
    ]:
        path = OUT_DIR / filename
        if path.is_file():
            st.markdown(f"#### {title}")
            st.dataframe(pd.read_csv(path), hide_index=True, use_container_width=True)

    if not comp_df.empty:
        with st.expander(f"Raw backtest records · {len(comp_df)} calls"):
            st.dataframe(comp_df, hide_index=True, use_container_width=True)

Writing app.py


##Live Streamlit Demo

In [28]:
import subprocess
import time

# Uvolní GPU paměť z notebooku, aby lokální Qwen v app.py měl plnou T4 VRAM
for _v in ["llm_model", "llm_tokenizer", "embedder", "reranker"]:
    if _v in globals():
        del globals()[_v]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

os.environ["EARNINGS_PROJECT_DIR"] = str(BASE_DIR.resolve())

if not Path("cloudflared").exists():
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "cloudflared"], check=True)
    subprocess.run(["chmod", "+x", "cloudflared"], check=True)

subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    env=os.environ.copy(),
    stdout=open("streamlit.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(3)

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print("Starting Cloudflare Tunnel...")
for line in tunnel.stdout:
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("\n=== LIVE STREAMLIT DASHBOARD URL ===")
        print(m.group(0))
        break

Starting Cloudflare Tunnel...

=== LIVE STREAMLIT DASHBOARD URL ===
https://intellectual-educated-obviously-bald.trycloudflare.com
